# 04 -- Cross-dataset transfer and ablations

Trains the six configurations of the paper at five seeds each on LAV-DF, evaluates them within-dataset (LAV-DF test split) and across datasets (all of FakeAVCeleb, never used for training or model selection), and tests each configuration against the full model.

**Runtime.** Each training run takes roughly 50-80 minutes on an RTX A5000 (24 GB), and the sweep is 30 runs, so plan on one to two days. The sweep writes `../data/runs/ablations/seed_results.json` after every run and **resumes** where it stopped if you re-run the cell.

## 4.1 Load everything from notebook 03

In [ ]:
# Runs notebook 03 in this kernel: defines the model, loaders and train_model, and trains the
# 'full' baseline once (seed 0). Notebook 03 must be saved under exactly this name.
%run 03_model_and_training.ipynb

## 4.2 The cross-dataset corpus (FakeAVCeleb)

Run notebooks 01 and 02 with `DATASET = "fakeavceleb"` first. The whole corpus is used as a held-out test set.

In [ ]:
from pathlib import Path
import numpy as np

CROSS_FEAT_ROOT = Path("../data/feats/fakeavceleb")
cross_files = sorted(CROSS_FEAT_ROOT.glob("*.npz"))
if not cross_files:
    raise FileNotFoundError(f"No features at {CROSS_FEAT_ROOT.resolve()}. Run notebooks 01 and 02 with "
                            "DATASET = 'fakeavceleb' first; there is no valid fallback for a cross-dataset test.")
print(f"FakeAVCeleb features: {len(cross_files)} clips")

In [ ]:
cross_meta = []
for f in cross_files:
    d = np.load(f, allow_pickle=True)
    cross_meta.append({"file": f, "label": int(d["label"]),
                       "subject": str(d["subject"]), "method": str(d["method"])})
labs = [m["label"] for m in cross_meta]
print(f"cross-dataset set: {len(cross_meta)} clips (real={labs.count(0)}, fake={labs.count(1)})")
assert len(set(labs)) == 2, "Cross set is single-class -- AUC would be undefined."
cross_loader = make_loader(cross_meta, cfg, train=False)

## 4.3 The configurations

In [ ]:
from dataclasses import replace
import json

ABLATIONS = {
    # tag             : overrides of the Cfg defaults in notebook 03
    "full":          {},                         # proposed configuration
    "no_sync":       {"lambda_sync": 0.0},       # synchrony objective removed
    "no_adv":        {"lambda_adv": 0.0},        # adversarial consistency term removed
    "no_crossmodal": {"use_crossmodal": False},  # cross-attention replaced by concatenation
    "no_temporal":   {"use_temporal": False},    # GRU removed (input projection only)
    "sync_strong":   {"lambda_sync": 1.0},       # five-fold stronger synchrony weight
    # "sync_weak":   {"lambda_sync": 0.1},       # run once during development, not replicated
}
print("configurations:", list(ABLATIONS))

## 4.4 Five seeds per configuration

Each run uses an explicit seed (weight initialisation, data order, augmentation and perturbation noise) and a data loader rebuilt for that seed. Results are saved after every run; re-running the cell resumes.

In [ ]:
import json, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

ABL_DIR = RUNS_ROOT / "ablations"; ABL_DIR.mkdir(parents=True, exist_ok=True)
SEED_FILE = ABL_DIR / "seed_results.json"
TAGS = list(ABLATIONS)
SEEDS = [0, 1, 2, 3, 4]

seed_results = json.loads(SEED_FILE.read_text()) if SEED_FILE.exists() else {}
for t in TAGS:
    seed_results.setdefault(t, {})              # {tag: {seed: {"cross_auc": ..., "within": {...}}}}

for s in SEEDS:
    tr_s = make_loader(train, cfg, True, seed=s)
    va_s = make_loader(val,   cfg, False)
    te_s = make_loader(test,  cfg, False)
    for tag in TAGS:
        if str(s) in seed_results[tag]:
            continue                            # already done: resume after an interruption
        c = replace(cfg, **ABLATIONS[tag])
        m, within = train_model(c, tr_s, va_s, te_s, tag=f"{tag}_seed{s}", seed=s, verbose=False)
        auc = evaluate(m, cross_loader)["auc"]
        seed_results[tag][str(s)] = {"cross_auc": auc, "within": within}
        SEED_FILE.write_text(json.dumps(seed_results, indent=1))
        print(f"  seed {s}  {tag:14s}  within AUC = {within['auc']:.4f}   cross AUC = {auc:.4f}")

## 4.5 Statistics

Welch's t-test of every configuration against the full model, Holm-corrected across the comparisons, plus a one-way ANOVA. With five seeds and a pooled SD of about 5 AUC points only differences of roughly 9 points or more are detectable, so read non-significant results as "no effect detected", not "no effect". The same computation is available offline as `scripts/stats_tests.py`.

In [ ]:
import numpy as np
from scipy import stats as st

cross = {t: [seed_results[t][str(s)]["cross_auc"] for s in SEEDS] for t in TAGS}
print(f"{'config':14s} {'n':>2s}  {'mean':>6s}  {'sd':>6s}   values")
for t, v in cross.items():
    a = np.array(v); print(f"{t:14s} {len(a):>2d}  {a.mean():.4f}  {a.std(ddof=1):.4f}   {[round(x, 4) for x in v]}")

full = np.array(cross["full"]); rows = []
for t, v in cross.items():
    if t == "full": continue
    a = np.array(v); rows.append((t, 100 * (a.mean() - full.mean()), st.ttest_ind(a, full, equal_var=False)[1]))
order = sorted(range(len(rows)), key=lambda i: rows[i][2]); adj, run = {}, 0.0
for rank, i in enumerate(order):
    run = max(run, min(1.0, (len(rows) - rank) * rows[i][2])); adj[rows[i][0]] = run
print("\n--- each configuration vs. full (Welch, Holm-corrected) ---")
for t, d, p in rows:
    print(f"  {t:14s} delta = {d:+6.2f} pts   p = {p:.3f}   Holm p = {adj[t]:.3f}   -> {'SIGNIFICANT' if adj[t] < 0.05 else 'not significant'}")
f, pa = st.f_oneway(*[np.array(v) for v in cross.values()])
print(f"\none-way ANOVA across configurations: F = {f:.2f}, p = {pa:.3f}")